[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/D.%20Integrated%20Tactical%20%26%20Pre-Planning%20Problems%20%28Connecting%20the%20Silos%29/34.%20The%20Hazard%20%26%20IMDG%20Segregation%20Problem/P34-Tier-9_executed.ipynb)

---


# 34. The Hazard and IMDG Segregation Problem

## Tier 9: A QAOA-Inspired Classical Demonstration

### Goal

This notebook models the assignment of hazardous-goods containers to cargo positions, expresses the assignment objective and constraints as a Quadratic Unconstrained Binary Optimization (QUBO) model, and explores assignments with a small stochastic classical search inspired by the parameterized structure of the Quantum Approximate Optimization Algorithm (QAOA).

The search code is a classical educational surrogate. It does **not** construct or execute a quantum circuit, simulate quantum amplitudes, or demonstrate quantum speedup. Its results are useful for studying the assignment model and the behavior of a randomized search, not for measuring quantum advantage.

### Inputs, outputs, and scope

- **Inputs:** container IDs and IMDG classes, cargo-position coordinates and hold IDs, and pairwise segregation requirements.
- **Output:** a one-to-one assignment of containers to positions, its modeled cost, and the number of segregation-rule violations.
- **Scope and assumptions:** the example uses a fixed set of 15 containers and 16 positions. Position coordinates are treated as points in three-dimensional Euclidean space, and the hold preference is a simple cost proportional to hold number. The model uses only the specified segregation rules and illustrative penalty weights; these weights are not regulatory distance recommendations. It does not model weight distribution, container dimensions, access sequences, detailed IMDG code interpretation, or operational uncertainty.

### Assignment model

Let $C$ be the set of containers and $P$ the set of cargo positions. The binary variable $x_{ip}$ is 1 when container $i$ is assigned to position $p$, and 0 otherwise. A feasible assignment puts each container in exactly one position and puts at most one container in each position:

$$
\sum_{p\in P}x_{ip}=1\quad(i\in C),
\qquad
\sum_{i\in C}x_{ip}\leq 1\quad(p\in P).
$$

For two containers assigned to positions $p$ and $q$, the modeled cost can include a distance-shortfall penalty, a different-hold penalty, or both. It also includes a hold preference cost. The QUBO adds quadratic penalties for violating the assignment constraints, allowing the model to be written as an unconstrained binary objective. The classical search in this notebook represents assignments directly and generates only assignments that already satisfy the one-container-per-position condition.

### What QAOA means, and what this demonstration does

QAOA is a hybrid quantum-classical algorithm. It encodes a binary objective in a cost Hamiltonian, alternates cost and mixing unitary operators for a chosen number of layers, measures the resulting quantum state, and uses a classical optimizer to adjust the circuit parameters. A genuine implementation requires a quantum circuit simulator or quantum hardware and a correctly specified measurement procedure. The classical routine in this notebook instead uses random assignments, parameter-dependent random changes, and greedy local improvements. Its parameters control that classical search; they are not QAOA circuit angles, and the routine provides no quantum-computing performance evidence.

### How to interpret the experiment

The random baseline and the search result are samples from stochastic classical procedures. Their comparison is an experiment on this instance, not a proof of optimality or a complexity result. No random seed is specified, so repeated runs can produce different samples and assignments. A zero violation count means only that the modeled distance and different-hold rules are satisfied. A nonzero count reports violations of those modeled rules. The modeled cost includes both the hold preference and weighted segregation penalties.

In [ ]:
from dataclasses import dataclass, field
from itertools import combinations
from typing import Any, Dict, List, Optional, Tuple

import random

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("default")
sns.set_palette("husl")

In [ ]:
@dataclass
class Container:
    """A container identified by its ID and IMDG class."""

    id: str
    imdg_class: str
    weight: float = 20.0
    qubit_index: Optional[int] = None


@dataclass
class Position:
    """A cargo position with hold membership and 3D coordinates."""

    id: str
    hold: int
    x: float
    y: float
    z: float
    capacity: float = 1.0
    qubit_indices: List[int] = field(default_factory=list)


@dataclass
class SegregationRequirement:
    """A pairwise distance or different-hold rule for two IMDG classes."""

    class1: str
    class2: str
    requirement: str
    min_distance: float
    different_hold: bool = False
    penalty_weight: float = 1.0


@dataclass
class QUBOCoefficient:
    """One linear or quadratic coefficient in a binary QUBO objective."""

    indices: Tuple[int, ...]
    value: float
    description: str


@dataclass
class QAOAConfig:
    """Settings for the classical QAOA-inspired search demonstration."""

    num_layers: int = 2
    max_iterations: int = 100
    learning_rate: float = 0.01
    shots: int = 1000
    optimization_method: str = "random_search"
    initial_parameters: Optional[List[float]] = None

In [ ]:
def create_quantum_scenario():
    """Create the fixed container, position, and segregation-rule instance."""

    container_configs = [
        ("QEXP01", "1.1"),
        ("QEXP02", "1.4S"),
        ("QEXP03", "1.4S"),
        ("QEXP04", "1.4S"),
        ("QFLM01", "3"),
        ("QFLM02", "3"),
        ("QFLM03", "3"),
        ("QFLM04", "3"),
        ("QCOR01", "8.1"),
        ("QCOR02", "8.1"),
        ("QCOR03", "8"),
        ("QGAS01", "2.1"),
        ("QGAS02", "2.3"),
        ("QMISC01", "9"),
        ("QMISC02", "9"),
    ]
    containers = [Container(container_id, imdg_class)
                  for container_id, imdg_class in container_configs]

    positions = []
    for hold in range(1, 5):
        for position_index in range(4):
            positions.append(
                Position(
                    id=f"QH{hold}_P{position_index + 1}",
                    hold=hold,
                    x=position_index * 3.0,
                    y=0.0,
                    z=(hold - 1) * 4.0,
                )
            )

    segregation_requirements = [
        SegregationRequirement("1.1", "1.4S", "Away From", 6.0, False, 3.0),
        SegregationRequirement("1.1", "3", "Away From", 8.0, False, 2.5),
        SegregationRequirement("1.1", "8", "Separated From", 0.0, True, 4.0),
        SegregationRequirement("1.1", "8.1", "Separated From", 0.0, True, 4.0),
        SegregationRequirement("1.1", "2.1", "Away From", 8.0, False, 2.5),
        SegregationRequirement("1.1", "2.3", "Away From", 8.0, False, 2.5),
        SegregationRequirement("1.1", "9", "Away From", 3.0, False, 1.5),
        SegregationRequirement("1.4S", "3", "Away From", 4.0, False, 2.0),
        SegregationRequirement("1.4S", "8", "Separated From", 0.0, True, 3.0),
        SegregationRequirement("1.4S", "8.1", "Separated From", 0.0, True, 3.0),
        SegregationRequirement("1.4S", "2.1", "Away From", 4.0, False, 2.0),
        SegregationRequirement("1.4S", "2.3", "Away From", 4.0, False, 2.0),
        SegregationRequirement("3", "8", "Away From", 3.0, False, 1.5),
        SegregationRequirement("3", "2.1", "No restriction", 0.0, False, 0.5),
        SegregationRequirement("3", "2.3", "Away From", 3.0, False, 1.5),
        SegregationRequirement("3", "9", "No restriction", 0.0, False, 0.5),
        SegregationRequirement("8", "2.3", "Away From", 3.0, False, 1.5),
        SegregationRequirement("8", "9", "Away From", 3.0, False, 1.0),
        SegregationRequirement("8.1", "2.3", "Away From", 3.0, False, 1.5),
        SegregationRequirement("8.1", "9", "Away From", 3.0, False, 1.0),
        SegregationRequirement("2.1", "2.3", "Away From", 3.0, False, 1.0),
        SegregationRequirement("2.1", "9", "No restriction", 0.0, False, 0.5),
        SegregationRequirement("2.3", "9", "Away From", 3.0, False, 1.0),
    ]

    return containers, positions, segregation_requirements


containers, positions, seg_reqs = create_quantum_scenario()

print("Classical QAOA-Inspired Demonstration Scenario")
print(f"Containers: {len(containers)}")
print(f"Positions: {len(positions)} (4 holds, 4 positions per hold)")
print(f"Segregation requirements: {len(seg_reqs)}")
print(f"Number of injective assignments: {np.prod(range(len(positions) - len(containers) + 1, len(positions) + 1)):,}")

print("\nContainer distribution by IMDG class:")
class_counts = {}
for container in containers:
    class_counts[container.imdg_class] = class_counts.get(container.imdg_class, 0) + 1
for imdg_class, count in sorted(class_counts.items()):
    print(f"  Class {imdg_class}: {count}")

print("\nPosition distribution by hold:")
hold_counts = {}
for position in positions:
    hold_counts[position.hold] = hold_counts.get(position.hold, 0) + 1
for hold, count in sorted(hold_counts.items()):
    print(f"  Hold {hold}: {count} positions")

## From assignments to a QUBO objective

A binary encoding assigns one binary variable to every container-position pair. With $|C|$ containers and $|P|$ positions, this one-hot assignment encoding uses $|C||P|$ binary variables. The index for $x_{ip}$ in the code is `i * num_positions + p`. A direct mapping from these variables to qubits therefore uses the same number of qubits; it is not the compact binary encoding of a position label.

For a feasible assignment, the hold preference cost for assigning container $i$ to position $p$ is $h_p=0.5\,\mathrm{hold}(p)$. For containers $i<j$ assigned to positions $p$ and $q$, let $d_{pq}$ be their Euclidean distance and let $r_{ij}$ denote their matching segregation requirement. The pair cost used by the code is

$$
g_{ijpq}=w_{ij}\left[10\max(0,d_{ij}^{\min}-d_{pq})+50\,\mathbf{1}\{\text{different hold required and }p,q\text{ share a hold}\}\right],
$$

where $d_{ij}^{\min}$ is the minimum distance specified by $r_{ij}$, and $w_{ij}$ is that requirement's penalty weight. A term is zero when there is no matching requirement; a requirement with no distance or different-hold restriction also contributes zero. The indicator is 1 when the stated condition is true and 0 otherwise. Thus a pair can incur both modeled penalties, while the reported violation count counts each failed distance or different-hold test separately.

For binary variables, the modeled assignment cost is

$$
H(x)=\sum_{i\in C}\sum_{p\in P}h_p x_{ip}
+\sum_{i<j}\sum_{p\in P}\sum_{q\in P}g_{ijpq}x_{ip}x_{jq}.
$$

The one-position-per-container condition is encoded by $A(\sum_p x_{ip}-1)^2$. The at-most-one-container-per-position condition is encoded by $A\sum_{i<j}x_{ip}x_{jp}$ for each position. Together, the QUBO objective is

$$
Q(x)=H(x)+A\sum_{i\in C}\left(\sum_{p\in P}x_{ip}-1\right)^2
+A\sum_{p\in P}\sum_{i<j}x_{ip}x_{jp},
$$

where $A$ is the positive assignment-penalty coefficient. Expanding the square uses $x_{ip}^2=x_{ip}$ for binary variables, producing a linear coefficient $-A$ for each variable and a quadratic coefficient $2A$ for each pair of positions belonging to the same container. The second penalty adds a quadratic coefficient $A$ when two different containers select the same position. The constant term $A|C|$ from the expansion is omitted from the stored coefficients because it shifts every objective value by the same amount and does not change which assignment minimizes it.

The code's `evaluate_solution` function evaluates $H(x)$ for a list of position indices and rejects invalid or non-injective assignments. `formulate_qubo` builds the linear and quadratic coefficients of $Q(x)$; it does not add the omitted constant. The stochastic search uses the direct assignment representation, so it generates feasible assignments without sampling arbitrary invalid bit strings. Pairwise costs are computed once for each container pair and position pair, then reused in repeated evaluations. Building these costs and the QUBO interactions requires work proportional to $|C|^2|P|^2$ in this implementation; the stored pair-cost table also uses $O(|C|^2|P|^2)$ space.

In [ ]:
class IMDGQUBOFormulator:
    """Build a binary QUBO and evaluate discrete container assignments."""

    def __init__(self, containers: List[Container], positions: List[Position],
                 requirements: List[SegregationRequirement]):
        if not containers or len(positions) < len(containers):
            raise ValueError("The instance needs containers and at least as many positions as containers.")

        self.containers = containers
        self.positions = positions
        self.requirements = requirements
        self.qubits_per_container = len(positions)
        self.total_qubits = len(containers) * self.qubits_per_container
        self.num_binary_variables = len(containers) * len(positions)
        self.assignment_penalty = 1_000_000.0
        self.linear_terms: List[QUBOCoefficient] = []
        self.quadratic_terms: List[QUBOCoefficient] = []
        self._requirement_lookup = {
            tuple(sorted((requirement.class1, requirement.class2))): requirement
            for requirement in requirements
        }
        self._pair_penalty_cache = {}

        num_containers = len(containers)
        num_positions = len(positions)
        self._pair_i, self._pair_j = np.triu_indices(num_containers, k=1)
        self._pair_cost_table = np.zeros(
            (num_containers, num_containers, num_positions, num_positions),
            dtype=float,
        )
        for container_i, container_j in combinations(range(num_containers), 2):
            for position_i in range(num_positions):
                for position_j in range(num_positions):
                    self._pair_cost_table[
                        container_i, container_j, position_i, position_j
                    ] = self._pair_penalty(
                        containers[container_i],
                        containers[container_j],
                        position_i,
                        position_j,
                    )

        print("QUBO Encoding")
        print(f"Binary assignment variables: {self.num_binary_variables}")
        print(f"Qubits in a direct one-hot mapping: {self.total_qubits}")
        print("The search uses the assignment representation directly and does not simulate a quantum state.")

    def _requirement_for_pair(self, container1: Container,
                              container2: Container) -> Optional[SegregationRequirement]:
        class_pair = tuple(sorted((container1.imdg_class, container2.imdg_class)))
        return self._requirement_lookup.get(class_pair)

    def calculate_distance_penalty(self, pos1_idx: int, pos2_idx: int,
                                   min_distance: float) -> float:
        """Return the shortfall penalty under the model's 3D Euclidean geometry."""
        pos1 = self.positions[pos1_idx]
        pos2 = self.positions[pos2_idx]
        distance = np.sqrt(
            (pos1.x - pos2.x) ** 2
            + (pos1.y - pos2.y) ** 2
            + (pos1.z - pos2.z) ** 2
        )
        return max(0.0, min_distance - distance) * 10.0

    def calculate_hold_penalty(self, pos1_idx: int, pos2_idx: int,
                               different_hold_required: bool) -> float:
        """Return the penalty for violating a different-hold requirement."""
        if not different_hold_required:
            return 0.0
        return 50.0 if self.positions[pos1_idx].hold == self.positions[pos2_idx].hold else 0.0

    def _pair_penalty(self, container1: Container, container2: Container,
                      pos1_idx: int, pos2_idx: int) -> float:
        pair_key = tuple(sorted((
            (container1.imdg_class, pos1_idx),
            (container2.imdg_class, pos2_idx),
        )))
        if pair_key in self._pair_penalty_cache:
            return self._pair_penalty_cache[pair_key]

        requirement = self._requirement_for_pair(container1, container2)
        if requirement is None:
            self._pair_penalty_cache[pair_key] = 0.0
            return 0.0

        penalty = 0.0
        if requirement.min_distance > 0:
            penalty += self.calculate_distance_penalty(
                pos1_idx, pos2_idx, requirement.min_distance
            )
        penalty += self.calculate_hold_penalty(
            pos1_idx, pos2_idx, requirement.different_hold
        )
        weighted_penalty = penalty * requirement.penalty_weight
        self._pair_penalty_cache[pair_key] = weighted_penalty
        return weighted_penalty

    def formulate_qubo(self) -> List[QUBOCoefficient]:
        """Construct linear and quadratic coefficients for the assignment QUBO."""
        linear = {}
        quadratic = {}
        descriptions = {}
        num_positions = len(self.positions)
        penalty = self.assignment_penalty

        def add_linear(index: int, value: float, description: str) -> None:
            linear[index] = linear.get(index, 0.0) + value
            descriptions[(index,)] = description

        def add_quadratic(index1: int, index2: int, value: float,
                          description: str) -> None:
            key = tuple(sorted((index1, index2)))
            quadratic[key] = quadratic.get(key, 0.0) + value
            descriptions[key] = description

        def variable_index(container_index: int, position_index: int) -> int:
            return container_index * num_positions + position_index

        # The expanded square penalizes selecting zero or multiple positions for a container.
        for container_index, container in enumerate(self.containers):
            indices = [variable_index(container_index, p)
                       for p in range(num_positions)]
            for index in indices:
                add_linear(index, -penalty, f"One-position constraint for {container.id}")
            for index1, index2 in combinations(indices, 2):
                add_quadratic(index1, index2, 2.0 * penalty,
                              f"One-position constraint for {container.id}")

            for position_index, position in enumerate(self.positions):
                index = variable_index(container_index, position_index)
                add_linear(index, position.hold * 0.5,
                           f"Hold preference for {container.id} at {position.id}")

        # Pairwise products penalize two containers assigned to the same position.
        for container1_index, container2_index in combinations(range(len(self.containers)), 2):
            container1 = self.containers[container1_index]
            container2 = self.containers[container2_index]
            for position1_index in range(num_positions):
                for position2_index in range(num_positions):
                    index1 = variable_index(container1_index, position1_index)
                    index2 = variable_index(container2_index, position2_index)
                    if position1_index == position2_index:
                        add_quadratic(index1, index2, penalty,
                                      "At-most-one-container-per-position constraint")
                    pair_cost = self._pair_penalty(
                        container1, container2, position1_index, position2_index
                    )
                    if pair_cost:
                        add_quadratic(index1, index2, pair_cost,
                                      f"Segregation cost for {container1.id} and {container2.id}")

        coefficients = [
            QUBOCoefficient((index,), value, descriptions[(index,)])
            for index, value in sorted(linear.items())
        ]
        coefficients.extend(
            QUBOCoefficient(indices, value, descriptions[indices])
            for indices, value in sorted(quadratic.items())
        )
        self.linear_terms = [coefficient for coefficient in coefficients
                             if len(coefficient.indices) == 1]
        self.quadratic_terms = [coefficient for coefficient in coefficients
                                if len(coefficient.indices) == 2]
        print(f"QUBO coefficients: {len(self.linear_terms)} linear, "
              f"{len(self.quadratic_terms)} quadratic")
        return coefficients

    def evaluate_solution(self, solution: List[int]) -> float:
        """Evaluate a feasible list of position indices using the modeled cost."""
        if len(solution) != len(self.containers):
            return float("inf")
        if any(not isinstance(index, (int, np.integer))
               or index < 0 or index >= len(self.positions) for index in solution):
            return float("inf")
        if len(set(solution)) != len(solution):
            return float("inf")

        assignment = np.asarray(solution, dtype=int)
        total_cost = sum(self.positions[pos_idx].hold * 0.5
                         for pos_idx in solution)
        pair_costs = self._pair_cost_table[
            self._pair_i,
            self._pair_j,
            assignment[self._pair_i],
            assignment[self._pair_j],
        ]
        total_cost += float(np.sum(pair_costs))
        return total_cost

    def count_violations(self, solution: List[int]) -> int:
        """Count violated distance and different-hold rules in a feasible assignment."""
        if not np.isfinite(self.evaluate_solution(solution)):
            raise ValueError("Violation counts require a valid, non-overlapping assignment.")

        violations = 0
        for i, j in combinations(range(len(self.containers)), 2):
            requirement = self._requirement_for_pair(self.containers[i], self.containers[j])
            if requirement is None:
                continue

            pos1 = self.positions[solution[i]]
            pos2 = self.positions[solution[j]]
            distance = np.sqrt(
                (pos1.x - pos2.x) ** 2
                + (pos1.y - pos2.y) ** 2
                + (pos1.z - pos2.z) ** 2
            )
            if requirement.min_distance > 0 and distance < requirement.min_distance:
                violations += 1
            if requirement.different_hold and pos1.hold == pos2.hold:
                violations += 1
        return violations

## Classical search procedure

The following procedure retains a layered, parameter-controlled search structure for teaching purposes, but all operations are classical. A starting assignment is generated by sampling distinct positions, so every candidate respects the one-container-per-position condition. At each layer, a parameter-controlled random mixing step proposes changes, and a greedy step accepts a trial move only when it reduces the modeled cost.

The outer loop perturbs the parameters and evaluates the resulting search behavior. This is not COBYLA or another named numerical optimizer: `optimization_method` documents the random-search procedure used here. The `shots` setting is retained as a configuration field for context, but it does not represent quantum measurements in this classical demonstration. The `learning_rate` field is also retained as configuration context; this random-search implementation does not use it to update parameters.

For $p$ layers, the parameter vector contains $2p$ values, paired as $(\gamma_\ell,\beta_\ell)$ for layers $\ell=1,\ldots,p$. In genuine QAOA these would control cost and mixing unitaries. Here they control random proposal probabilities and classical local improvement only. In particular, the mixing probability is clipped to $[0,1]$ from $\beta_\ell/\pi$, and the number of local proposals is $1+\lfloor 2\,\operatorname{clip}(\gamma_\ell/(2\pi),0,1)\rfloor$. The code reports the best feasible assignment encountered by this stochastic search; it does not certify a global optimum.

At each parameter-search iteration, the procedure estimates a mean cost from transformed random assignments. It separately transforms a batch of five candidates and retains the lowest-cost assignment found. If the current parameter vector is $\theta$, the estimate is

$$
\widehat{J}(\theta)=\frac{1}{m}\sum_{s=1}^{m}H\bigl(T(\pi_s;\theta)\bigr),
$$

where $m=\min(30,\max(10,2|C|))$, each $\pi_s$ is a uniformly sampled injective assignment, and $T$ is the layered classical transformation implemented by `_apply_quantum_transformation`. The objective estimate is stochastic because its assignments and proposed moves are random. After evaluating the current parameters, each parameter is perturbed by Gaussian noise with standard deviation $0.1$, clipped to $[0,2\pi]`, and accepted if its estimated cost improves or with probability $0.1$. The best assignment and the parameter vector with the best objective estimate are tracked separately.

For $I$ outer iterations, $2p$ parameters, and $m$ objective samples, the number of transformed samples grows on the order of $I(2p+1)m$. Each transformation applies $p$ layers across the containers and can make several full assignment-cost evaluations. The precomputed pair-cost table makes each such cost evaluation a vectorized lookup and sum over the $|C|(|C|-1)/2$ container pairs. The procedure remains stochastic and has no optimality guarantee.

In [ ]:
class QuantumQAOA:
    """Run a classical randomized search inspired by QAOA's layered structure."""

    def __init__(self, qubo_formulator: IMDGQUBOFormulator, config: QAOAConfig):
        if config.num_layers < 1 or config.max_iterations < 1:
            raise ValueError("num_layers and max_iterations must be positive.")
        self.qubo_formulator = qubo_formulator
        self.config = config
        self.coefficients = qubo_formulator.formulate_qubo()
        num_parameters = 2 * config.num_layers
        if config.initial_parameters is None:
            self.parameters = [random.uniform(0.0, np.pi)
                               for _ in range(num_parameters)]
        else:
            if len(config.initial_parameters) != num_parameters:
                raise ValueError("initial_parameters must contain two values per layer.")
            self.parameters = config.initial_parameters.copy()

        self.optimization_history = []
        self.best_solution = None
        self.best_cost = float("inf")

        print("Classical QAOA-Inspired Search Initialized")
        print(f"Layers: {config.num_layers}")
        print(f"Binary assignment variables: {qubo_formulator.num_binary_variables}")
        print(f"Maximum iterations: {config.max_iterations}")

    def _random_feasible_solution(self) -> List[int]:
        """Sample distinct positions so the proposed assignment is feasible."""
        return random.sample(
            range(len(self.qubo_formulator.positions)),
            len(self.qubo_formulator.containers),
        )

    def quantum_circuit_simulation(self, parameters: List[float],
                                   solution_sample: List[List[int]]) -> List[float]:
        """Apply the classical layered search to each sampled assignment."""
        return [
            self.qubo_formulator.evaluate_solution(
                self._apply_quantum_transformation(solution, parameters)
            )
            for solution in solution_sample
        ]

    def _apply_quantum_transformation(self, solution: List[int],
                                      parameters: List[float]) -> List[int]:
        """Use parameter-controlled random proposals and greedy cost improvement."""
        transformed = solution.copy()
        num_positions = len(self.qubo_formulator.positions)

        for layer in range(self.config.num_layers):
            gamma = parameters[2 * layer]
            beta = parameters[2 * layer + 1]
            mixing_probability = min(1.0, max(0.0, beta / np.pi))

            for container_index in range(len(transformed)):
                if random.random() < mixing_probability:
                    other_indices = [index for index in range(len(transformed))
                                     if index != container_index]
                    if other_indices:
                        other_index = random.choice(other_indices)
                        transformed[container_index], transformed[other_index] = (
                            transformed[other_index], transformed[container_index]
                        )

                current_cost = self.qubo_formulator.evaluate_solution(transformed)
                proposal_count = 1 + int(2 * min(1.0, max(0.0, gamma / (2.0 * np.pi))))
                for _ in range(proposal_count):
                    available_positions = set(range(num_positions)) - set(transformed)
                    if not available_positions:
                        break
                    candidate = transformed.copy()
                    candidate[container_index] = random.choice(list(available_positions))
                    candidate_cost = self.qubo_formulator.evaluate_solution(candidate)
                    if candidate_cost < current_cost:
                        transformed = candidate
                        current_cost = candidate_cost

        return transformed

    def objective_function(self, parameters: List[float]) -> float:
        """Estimate search quality by the mean cost of transformed samples."""
        sample_count = min(30, max(10, len(self.qubo_formulator.containers) * 2))
        solution_samples = [self._random_feasible_solution()
                            for _ in range(sample_count)]
        costs = self.quantum_circuit_simulation(parameters, solution_samples)
        return float(np.mean(costs))

    def optimize(self) -> Dict[str, Any]:
        """Search parameter settings and retain the best feasible assignment found."""
        best_objective = float("inf")
        best_parameters = self.parameters.copy()

        print("\nStarting the classical parameter-search experiment")
        print(f"Containers: {len(self.qubo_formulator.containers)}")
        print(f"Positions: {len(self.qubo_formulator.positions)}")
        print(f"Parameter search: {self.config.optimization_method}")

        for iteration in range(self.config.max_iterations):
            current_cost = self.objective_function(self.parameters)
            candidate_solutions = [self._random_feasible_solution()
                                   for _ in range(5)]
            transformed_solutions = [
                self._apply_quantum_transformation(solution, self.parameters)
                for solution in candidate_solutions
            ]
            candidate_costs = [self.qubo_formulator.evaluate_solution(solution)
                               for solution in transformed_solutions]
            best_index = int(np.argmin(candidate_costs))
            if candidate_costs[best_index] < self.best_cost:
                self.best_cost = candidate_costs[best_index]
                self.best_solution = transformed_solutions[best_index].copy()

            if current_cost < best_objective:
                best_objective = current_cost
                best_parameters = self.parameters.copy()

            self.optimization_history.append({
                "iteration": iteration,
                "cost": current_cost,
                "best_cost": self.best_cost,
                "parameters": self.parameters.copy(),
            })

            if (iteration + 1) % 20 == 0 or iteration == 0:
                print(f"Iteration {iteration + 1:3d}: "
                      f"Mean sample cost = {current_cost:.2f}, "
                      f"Best assignment cost = {self.best_cost:.2f}")

            for parameter_index in range(len(self.parameters)):
                proposal = self.parameters.copy()
                proposal[parameter_index] = np.clip(
                    proposal[parameter_index] + np.random.normal(0.0, 0.1),
                    0.0,
                    2.0 * np.pi,
                )
                proposal_cost = self.objective_function(proposal)
                if proposal_cost < current_cost or random.random() < 0.1:
                    self.parameters = proposal
                    current_cost = proposal_cost

        if self.best_solution is None:
            raise RuntimeError("The search did not produce a feasible assignment.")

        final_cost = self.qubo_formulator.evaluate_solution(self.best_solution)
        final_violations = self.qubo_formulator.count_violations(self.best_solution)
        assert np.isclose(final_cost, self.best_cost)

        return {
            "best_solution": self.best_solution.copy(),
            "best_cost": final_cost,
            "best_parameters": best_parameters,
            "final_violations": final_violations,
            "optimization_history": self.optimization_history,
            "iterations": self.config.max_iterations,
            "solution_quality": {
                "violations": final_violations,
                "cost": final_cost,
                "containers_placed": len(self.best_solution),
                "total_containers": len(self.qubo_formulator.containers),
                "success_rate": 100.0,
            },
        }

## Run the search and compare with random feasible assignments

The baseline samples feasible assignments uniformly from the available position labels using the same injective-assignment representation. It is a descriptive reference for this experiment, not a guarantee that either method finds an optimum. The reported cost is the modeled objective, and the violation count concerns only the specified distance and different-hold rules.

In [ ]:
print("IMDG Segregation: Classical QAOA-Inspired Search")
print("=" * 68)
print(f"Instance: {len(containers)} containers and {len(positions)} positions")
print("This cell runs a classical stochastic search; it does not execute a quantum circuit.")

qubo_formulator = IMDGQUBOFormulator(containers, positions, seg_reqs)
qaoa_config = QAOAConfig(
    num_layers=3,
    max_iterations=100,
    learning_rate=0.01,
    shots=100,
    optimization_method="random_search",
    initial_parameters=None,
)
qaoa = QuantumQAOA(qubo_formulator, qaoa_config)
qaoa_result = qaoa.optimize()

print("\nSearch result")
print(f"Modeled assignment cost: {qaoa_result['best_cost']:.2f}")
print(f"Modeled segregation-rule violations: {qaoa_result['final_violations']}")
print(f"Containers placed: {qaoa_result['solution_quality']['containers_placed']} / "
      f"{qaoa_result['solution_quality']['total_containers']}")
print(f"Search iterations: {qaoa_result['iterations']}")

random_costs = []
random_violations = []
for _ in range(100):
    random_solution = random.sample(range(len(positions)), len(containers))
    random_costs.append(qubo_formulator.evaluate_solution(random_solution))
    random_violations.append(qubo_formulator.count_violations(random_solution))

average_random_cost = float(np.mean(random_costs))
average_random_violations = float(np.mean(random_violations))
minimum_random_cost = float(np.min(random_costs))
minimum_random_violations = int(np.min(random_violations))

print("\nRandom feasible-assignment baseline (100 samples)")
print(f"Average modeled cost: {average_random_cost:.2f}")
print(f"Average modeled violations: {average_random_violations:.2f}")
print(f"Lowest sampled cost: {minimum_random_cost:.2f}")
print(f"Lowest sampled violation count: {minimum_random_violations}")

if not np.isclose(average_random_cost, 0.0):
    cost_reduction = 100.0 * (average_random_cost - qaoa_result["best_cost"]) / average_random_cost
    print(f"Cost difference relative to baseline average: {cost_reduction:.1f}%")
else:
    print("Cost difference relative to baseline average: undefined because the average is zero.")

print("\nBest assignment found by the classical search")
for container, position_index in zip(containers, qaoa_result["best_solution"]):
    position = positions[position_index]
    print(f"{container.id} (Class {container.imdg_class}) -> "
          f"{position.id} (Hold {position.hold})")

assert len(set(qaoa_result["best_solution"])) == len(containers)
assert np.isclose(
    qubo_formulator.evaluate_solution(qaoa_result["best_solution"]),
    qaoa_result["best_cost"],
)

## Visualize the experiment

The plots summarize the stochastic search history, the sampled random-baseline costs, the modeled violation counts, and the final container locations. The placement plot shows the $x$-$z$ projection of the supplied three-dimensional coordinates; the $z$ coordinate separates the holds in this instance, while the $y$ coordinate is constant. Color identifies the IMDG class, as indicated by the legend. The figures describe this particular run and do not establish quantum speedup, scalability, or global optimality.

In [ ]:
def visualize_qaoa_results(qaoa_result: Dict[str, Any], qaoa: QuantumQAOA,
                            random_costs: List[float],
                            random_violations: List[int],
                            containers: List[Container],
                            positions: List[Position]) -> None:
    """Plot search behavior, baseline distributions, and the final placement."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))

    history = qaoa_result["optimization_history"]
    iterations = [record["iteration"] + 1 for record in history]
    sample_costs = [record["cost"] for record in history]
    best_costs = [record["best_cost"] for record in history]
    axes[0, 0].plot(iterations, sample_costs, alpha=0.65, label="Mean sample cost")
    axes[0, 0].plot(iterations, best_costs, linewidth=2, label="Best assignment cost")
    axes[0, 0].set_xlabel("Iteration")
    axes[0, 0].set_ylabel("Modeled cost")
    axes[0, 0].set_title("Classical Search History")
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].hist(random_costs, bins=20, alpha=0.75, color="darkorange",
                    edgecolor="black", label="Random feasible assignments")
    axes[0, 1].axvline(qaoa_result["best_cost"], color="navy", linestyle="--",
                       linewidth=2, label="Best search assignment")
    axes[0, 1].set_xlabel("Modeled cost")
    axes[0, 1].set_ylabel("Sample count")
    axes[0, 1].set_title("Cost Comparison")
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3, axis="y")

    mean_violations = float(np.mean(random_violations))
    violation_labels = ["Random baseline\n(mean)", "Search assignment"]
    violation_values = [mean_violations, qaoa_result["final_violations"]]
    bars = axes[1, 0].bar(violation_labels, violation_values,
                          color=["darkorange", "seagreen"], alpha=0.8)
    axes[1, 0].set_ylabel("Modeled rule violations")
    axes[1, 0].set_title("Segregation-Rule Violations")
    axes[1, 0].grid(True, alpha=0.3, axis="y")
    for bar, value in zip(bars, violation_values):
        axes[1, 0].text(bar.get_x() + bar.get_width() / 2,
                        bar.get_height(), f"{value:.2f}",
                        ha="center", va="bottom")

    class_colors = {
        "1.1": "darkred", "1.4S": "red", "3": "darkorange",
        "8": "purple", "8.1": "darkviolet", "2.1": "green",
        "2.3": "darkgreen", "9": "royalblue",
    }
    assignment = qaoa_result["best_solution"]
    ax = axes[1, 1]
    for container_index, position_index in enumerate(assignment):
        position = positions[position_index]
        container = containers[container_index]
        color = class_colors.get(container.imdg_class, "gray")
        ax.scatter(position.x, position.z, s=120, color=color,
                   edgecolors="black", label=f"Class {container.imdg_class}")
        ax.annotate(container.id, (position.x, position.z),
                    xytext=(4, 4), textcoords="offset points", fontsize=7)

    ax.set_title("Assigned Positions (X-Z Projection; Color = IMDG Class)")
    ax.set_xlabel("X coordinate (model units)")
    ax.set_ylabel("Z coordinate (model units)")
    ax.grid(True, alpha=0.3)
    ax.set_aspect("equal", adjustable="datalim")

    handles, labels = ax.get_legend_handles_labels()
    unique_labels = dict(zip(labels, handles))
    if unique_labels:
        ax.legend(unique_labels.values(), unique_labels.keys(),
                  title="IMDG class", fontsize=8)

    fig.suptitle("IMDG Assignment: Classical Search Experiment", fontsize=15)
    fig.tight_layout()
    plt.show()


visualize_qaoa_results(
    qaoa_result,
    qaoa,
    random_costs,
    random_violations,
    containers,
    positions,
)

## Interpretation and limitations

The experiment illustrates how a discrete container-position assignment can be represented with binary variables, how pairwise segregation costs become quadratic interactions, and how assignment constraints can be incorporated into a QUBO objective. The stochastic search and random baseline provide two ways to generate feasible assignments and compare their modeled costs on the same instance.

A low modeled cost is not automatically a safe operational plan: the result depends on the supplied rules, coordinates, and penalty weights. A zero modeled violation count indicates satisfaction only of the encoded distance and different-hold checks. The model does not establish compliance with all applicable regulations.

This notebook does not provide evidence of quantum speedup or polynomial-time solution of an NP-hard problem. The search is classical, stochastic, and has no optimality guarantee. The QUBO representation is a mathematical formulation; it is not itself a quantum circuit. A genuine QAOA experiment would require encoding this objective into cost and mixing operators, sampling measurement outcomes from a quantum circuit simulation or device, and evaluating performance with an appropriate classical comparison. Those steps are outside the implemented classical demonstration.